# File Handling in Python

## Why This Matters
Nearly every real program reads or writes files — configs, logs, data exports, reports. Python has excellent built-in tools for text files, plus the `json`, `csv`, `os`, and `pathlib` modules for structured data and paths.

## Table of Contents
1. `open()` modes — r, w, a, rb, wb
2. The `with` statement — why it matters
3. Reading files — read / readline / readlines
4. Writing files — write / writelines
5. File positioning — seek / tell
6. Working with JSON
7. Working with CSV
8. `os.path` basics
9. `pathlib.Path` — the modern alternative
10. Common Mistakes
11. Exercises
12. Solutions
13. Mini-Project: Student Grade Manager


## 1. open() Modes

`open(filename, mode)` opens a file. The **mode** controls what you can do and what happens to existing content.

| Mode | Meaning | File must exist? | Creates file? | Truncates? |
|------|---------|-------------------|--------------|------------|
| `'r'` | Read text | Yes | No | No |
| `'w'` | Write text | No | Yes | Yes (clears!) |
| `'a'` | Append text | No | Yes | No |
| `'rb'` | Read binary | Yes | No | No |
| `'wb'` | Write binary | No | Yes | Yes |
| `'r+'` | Read+Write | Yes | No | No |

**Key rules:**
- Default mode is `'r'` (read text)
- `'w'` is destructive — it wipes the file before writing
- Use `'a'` to add to an existing file without erasing it
- Use `'b'` for images, PDFs, or any non-text file


In [ ]:
import os

# Write a sample file to work with throughout this notebook
f = open("sample.txt", "w")   # 'w' creates the file (or wipes it)
f.write("Line 1: Hello, World!\n")
f.write("Line 2: Python file handling\n")
f.write("Line 3: Reading and writing files\n")
f.close()  # IMPORTANT: always close the file when done

print("File written. Size:", os.path.getsize("sample.txt"), "bytes")

# Append mode: add to the file without erasing
f = open("sample.txt", "a")
f.write("Line 4: Appended later\n")
f.close()
print("Line appended. New size:", os.path.getsize("sample.txt"), "bytes")


## 2. The `with` Statement — Why It Matters

The `with` statement is Python's way of saying **"manage this resource safely"**. When used with file objects, it guarantees the file is closed when the block exits — even if an exception occurs.

**Analogy:** It's like borrowing a book from the library with an automatic return system. You don't have to remember to return it — the system handles it.

**Key rules:**
- Always prefer `with open(...)` over manual `open()` / `close()`
- The file is closed automatically when you leave the `with` block
- Prevents resource leaks (too many open file handles)
- Works with any object that has `__enter__` and `__exit__` methods


In [ ]:
# The SAFE way: using 'with' — file is always closed
with open("sample.txt", "r") as f:
    content = f.read()
    print("File is open inside with block:", not f.closed)

# After the 'with' block, file is automatically closed
print("File is closed after with block:", f.closed)

# Compare: the RISKY way (easy to forget f.close())
# f = open("sample.txt")
# content = f.read()
# ... if an exception happens here, f.close() never runs!
# f.close()


In [ ]:
# You can open multiple files in one 'with' statement
with open("sample.txt", "r") as source, open("output.txt", "w") as dest:
    for line in source:
        dest.write(line.upper())  # Copy with uppercase transformation

# Verify the output
with open("output.txt", "r") as f:
    print(f.read())


## 3. Reading Files — read / readline / readlines

Three methods, three use cases:

- **`read()`** — reads the ENTIRE file into a single string. Simple, but uses lots of memory for large files.
- **`readline()`** — reads ONE line at a time. Useful for processing a file line by line without loading everything.
- **`readlines()`** — reads ALL lines into a LIST of strings. Each string includes the `\n` newline character.
- **Iterating directly** — the most Pythonic way; memory-efficient like `readline()` but simpler syntax.


In [ ]:
# 1. read() — entire file as one string
with open("sample.txt", "r") as f:
    content = f.read()
print("read():")
print(repr(content))  # repr shows \n explicitly
print()

# 2. readlines() — list of lines (each line ends with \n)
with open("sample.txt", "r") as f:
    lines = f.readlines()
print("readlines():")
print(lines)  # ['Line 1: Hello, World!\n', 'Line 2: ...', ...]
print()

# 3. readline() — one line at a time
with open("sample.txt", "r") as f:
    first  = f.readline()  # reads first line
    second = f.readline()  # reads second line
print("readline():")
print(repr(first))
print(repr(second))


In [ ]:
# Best practice: iterate over the file object directly
# Memory efficient — only loads one line at a time
print("Direct iteration (most Pythonic):")
with open("sample.txt", "r") as f:
    for line_num, line in enumerate(f, start=1):
        # .strip() removes the trailing \n
        print(f"  [{line_num}] {line.strip()}")


## 4. Writing Files — write / writelines

- **`write(string)`** — writes a single string. Does NOT add a newline automatically.
- **`writelines(list)`** — writes a list of strings. Also does NOT add newlines automatically.

**Key rules:**
- Both methods return the number of characters written
- You must manually add `\n` where you want line breaks
- Use `'w'` mode to start fresh, `'a'` to append


In [ ]:
# write() — add \n manually
with open("write_demo.txt", "w") as f:
    chars = f.write("First line\n")   # returns number of chars written
    f.write("Second line\n")
    f.write("Third line\n")
    print(f"Characters in first write: {chars}")

# writelines() — provide a list, each item must have its own \n
lines = ["Alpha\n", "Beta\n", "Gamma\n"]
with open("writelines_demo.txt", "w") as f:
    f.writelines(lines)

# Verify
with open("writelines_demo.txt", "r") as f:
    print(f.read())


## 5. File Positioning — seek / tell

Files have an internal **cursor** that tracks where the next read/write will happen. Think of it like a cursor in a text editor.

- **`tell()`** — returns the current cursor position (in bytes from the start)
- **`seek(position)`** — moves the cursor to a specific position
- **`seek(0)`** — rewind to the beginning
- **`seek(0, 2)`** — move to the end of the file


In [ ]:
with open("sample.txt", "r") as f:
    print("Position at start:", f.tell())       # 0

    first_line = f.readline()
    print(f"After reading first line: {f.tell()}")  # position after \n
    print(f"First line: {first_line.strip()!r}")

    # Jump back to the beginning
    f.seek(0)
    print("\nAfter seek(0):", f.tell())          # 0 again

    # Read first line again
    first_again = f.readline()
    print(f"Re-read first line: {first_again.strip()!r}")

    # Jump to end to get file size
    f.seek(0, 2)  # 2 = from end of file
    print("\nFile size (bytes):", f.tell())


## 6. Working with JSON

JSON (JavaScript Object Notation) is the most common format for structured data exchange. Python's `json` module maps JSON directly to Python dicts, lists, strings, and numbers.

| Function | Direction | Works with |
|----------|-----------|------------|
| `json.dump(obj, file)` | Python → JSON file | file object |
| `json.load(file)` | JSON file → Python | file object |
| `json.dumps(obj)` | Python → JSON string | string |
| `json.loads(string)` | JSON string → Python | string |

**Memory trick:** `dump`/`load` work with files (no 's'). `dumps`/`loads` work with strings ('s' = string).


In [ ]:
import json

# Python object to save
data = {
    "name": "Alice",
    "age": 30,
    "scores": [95, 87, 92],
    "active": True,
    "address": None
}

# Write to JSON file
with open("data.json", "w") as f:
    json.dump(data, f, indent=2)  # indent makes it human-readable

# Read it back
with open("data.json", "r") as f:
    loaded = json.load(f)

print("Loaded data:", loaded)
print("Type:", type(loaded))  # dict
print("Scores:", loaded["scores"])


In [ ]:
# dumps / loads — working with JSON as a string (useful for APIs, logging)
config = {"host": "localhost", "port": 8080, "debug": False}

# Convert to JSON string
json_str = json.dumps(config, indent=2)
print("JSON string:")
print(json_str)
print("Type:", type(json_str))  # str

# Parse JSON string back to dict
parsed = json.loads(json_str)
print("\nParsed back:", parsed)
print("Port:", parsed["port"])  # 8080


## 7. Working with CSV

CSV (Comma-Separated Values) is the standard format for tabular data. Python's `csv` module handles quoting, escaping, and different delimiters automatically.

**Key classes:**
- `csv.reader` — reads rows as lists
- `csv.writer` — writes lists as rows
- `csv.DictReader` — reads rows as dicts (using header row as keys)
- `csv.DictWriter` — writes dicts as rows


In [ ]:
import csv

# Write CSV with csv.writer (rows as lists)
rows = [
    ["Name",  "Score", "Grade"],
    ["Alice", 92,      "A"],
    ["Bob",   78,      "C"],
    ["Carol", 85,      "B"],
]

# newline='' is required on Windows to prevent extra blank lines
with open("students.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)  # Write all rows at once

# Read it back with csv.reader
with open("students.csv", "r") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)  # Each row is a list of strings


In [ ]:
# DictWriter — write dicts as rows (cleaner for structured data)
students = [
    {"name": "Alice", "score": 92, "grade": "A"},
    {"name": "Bob",   "score": 78, "grade": "C"},
    {"name": "Carol", "score": 85, "grade": "B"},
]

with open("students_dict.csv", "w", newline="") as f:
    fieldnames = ["name", "score", "grade"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()  # Write the header row
    writer.writerows(students)

# DictReader — read rows as dicts (keys come from header)
print("Reading with DictReader:")
with open("students_dict.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"  {row['name']}: score={row['score']}, grade={row['grade']}")


## 8. os.path Basics

The `os.path` module provides functions to work with file and directory paths in a platform-independent way.

**Key functions:**
- `os.path.exists(path)` — check if a path exists
- `os.path.join(dir, file)` — combine path components safely (handles `/` vs `\`)
- `os.path.dirname(path)` — get the directory part of a path
- `os.path.basename(path)` — get the filename part of a path
- `os.path.splitext(path)` — split into `(name, extension)`
- `os.path.abspath(path)` — convert to absolute path


In [ ]:
import os

path = "/home/user/documents/report.pdf"

print("exists()  :", os.path.exists("sample.txt"))   # True (we created it)
print("exists()  :", os.path.exists("missing.txt"))   # False
print()
print("dirname() :", os.path.dirname(path))           # /home/user/documents
print("basename():", os.path.basename(path))          # report.pdf
print("splitext():", os.path.splitext(path))          # ('/home/user/documents/report', '.pdf')
print()

# join() is the safe way to build paths — handles OS differences
base_dir = "/home/user"
sub_dir  = "documents"
filename = "notes.txt"
full_path = os.path.join(base_dir, sub_dir, filename)
print("join()    :", full_path)  # /home/user/documents/notes.txt


## 9. pathlib.Path — The Modern Alternative

`pathlib` (Python 3.4+) offers an object-oriented approach to paths. Instead of string functions, you use a `Path` object with methods and the `/` operator for joining.

**Why prefer pathlib over os.path:**
- Cleaner syntax: `path / "subdir" / "file.txt"` instead of `os.path.join(...)`
- Methods directly on the object: `path.exists()`, `path.read_text()`
- Can read/write files without `open()` for simple cases
- Cross-platform by default


In [ ]:
from pathlib import Path

# Create a Path object
p = Path("sample.txt")

# Properties and methods
print("Path:     ", p)
print("Name:     ", p.name)        # sample.txt
print("Stem:     ", p.stem)        # sample (without extension)
print("Suffix:   ", p.suffix)      # .txt
print("Parent:   ", p.parent)      # . (current directory)
print("Exists:   ", p.exists())    # True
print("Absolute: ", p.resolve())   # full absolute path


In [ ]:
# Read and write with pathlib — no explicit open() needed for simple cases
p = Path("pathlib_demo.txt")

# Write text
p.write_text("Hello from pathlib!\nLine 2\nLine 3\n")

# Read text
content = p.read_text()
print(content)

# Join paths with / operator (intuitive!)
base = Path("/home/user")
full = base / "documents" / "report.txt"  # Clean path joining!
print("Joined path:", full)

# List all .txt files in current directory
txt_files = list(Path(".").glob("*.txt"))
print("\n.txt files here:", [f.name for f in txt_files])


## 10. Common Mistakes


In [ ]:
# MISTAKE 1: Using 'w' mode when you meant 'a' — accidentally erasing the file
with open("important.txt", "w") as f:
    f.write("Important data\n")

# This ERASES everything and starts fresh — is that what you wanted?
with open("important.txt", "w") as f:
    f.write("New content\n")  # The original data is GONE!

with open("important.txt") as f:
    print("File now contains:", f.read())

# Fix: use 'a' to append, or check first
with open("important.txt", "a") as f:
    f.write("Additional content (appended)\n")

with open("important.txt") as f:
    print("After append:", f.read())


In [ ]:
# MISTAKE 2: Not handling FileNotFoundError
try:
    with open("nonexistent_file.txt", "r") as f:
        content = f.read()
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Always handle this when the file might not exist!")

# The safe pattern: check first, or handle the exception
filename = "maybe_exists.txt"
if os.path.exists(filename):
    with open(filename) as f:
        print(f.read())
else:
    print(f"'{filename}' does not exist — skipping.")


In [ ]:
# MISTAKE 3: Forgetting newline='' in CSV writing on Windows
# Without newline='', csv.writer adds an extra blank line between rows

# WRONG (on Windows, produces blank lines between rows):
# with open('bad.csv', 'w') as f:      # missing newline=''
#     writer = csv.writer(f)
#     writer.writerow(['a', 'b', 'c'])

# CORRECT:
with open("good.csv", "w", newline="") as f:  # newline='' prevents double newlines
    writer = csv.writer(f)
    writer.writerow(["name", "value"])
    writer.writerow(["alpha", 1])
    writer.writerow(["beta",  2])

print("CSV written correctly.")


## 11. Exercises

### Exercise 1 (Easy): Count lines and words
Write a function `file_stats(filename)` that opens a text file and returns a dict with `lines`, `words`, and `characters` counts.


In [ ]:
# Your code here
def file_stats(filename):
    pass  # Replace with your implementation

# Test it
# stats = file_stats("sample.txt")
# print(stats)  # e.g., {'lines': 4, 'words': 17, 'characters': 103}


### Exercise 2 (Medium): CSV to JSON converter
Write a function `csv_to_json(csv_path, json_path)` that reads a CSV file (with a header row) and writes it as a JSON file — a list of dicts, one per row.


In [ ]:
# Your code here
def csv_to_json(csv_path, json_path):
    pass  # Replace with your implementation

# Test it
# csv_to_json("students_dict.csv", "students_output.json")
# with open("students_output.json") as f:
#     print(f.read())


### Exercise 3 (Hard): Log file analyzer
Given a multiline log string (write it to a file first), write code that reads the file and produces a summary: how many lines contain each log level (INFO, WARNING, ERROR). Return a dict like `{'INFO': 5, 'WARNING': 2, 'ERROR': 1}`.


In [ ]:
# Sample log data to write to a file first
log_data = """2024-01-01 10:00:01 INFO  Server started
2024-01-01 10:00:05 INFO  Database connected
2024-01-01 10:01:23 WARNING High memory usage detected
2024-01-01 10:02:45 INFO  Request processed successfully
2024-01-01 10:03:01 ERROR Failed to connect to cache
2024-01-01 10:03:15 INFO  Retry successful
2024-01-01 10:04:00 WARNING Slow query detected
"""

with open("app.log", "w") as f:
    f.write(log_data)

# Your code here: analyze the log file
def analyze_log(filename):
    pass  # Return {'INFO': n, 'WARNING': n, 'ERROR': n}

# print(analyze_log("app.log"))


## 12. Solutions


In [ ]:
# SOLUTION 1 - try it yourself first!
def file_stats(filename):
    with open(filename, "r") as f:
        content = f.read()
    lines = content.splitlines()
    words = content.split()
    return {
        "lines":      len(lines),
        "words":      len(words),
        "characters": len(content),
    }

print(file_stats("sample.txt"))


In [ ]:
# SOLUTION 2 - try it yourself first!
def csv_to_json(csv_path, json_path):
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        rows = list(reader)  # list of OrderedDicts
    with open(json_path, "w") as f:
        json.dump(rows, f, indent=2)
    print(f"Converted {len(rows)} rows from {csv_path} to {json_path}")

csv_to_json("students_dict.csv", "students_output.json")
with open("students_output.json") as f:
    print(f.read())


In [ ]:
# SOLUTION 3 - try it yourself first!
def analyze_log(filename):
    counts = {"INFO": 0, "WARNING": 0, "ERROR": 0}
    with open(filename, "r") as f:
        for line in f:
            for level in counts:
                if level in line:
                    counts[level] += 1
                    break  # Each line has at most one level
    return counts

print(analyze_log("app.log"))


## 13. Mini-Project: Student Grade Manager

Build a complete grade management system that:
1. Writes student records to a CSV file
2. Reads them back and computes statistics
3. Saves a summary report to JSON


In [ ]:
import csv, json, os
from pathlib import Path

# ── Step 1: Write student records to CSV ─────────────────────────────────────
students = [
    {"name": "Alice",   "math": 92, "science": 88, "english": 95},
    {"name": "Bob",     "math": 74, "science": 81, "english": 68},
    {"name": "Carol",   "math": 88, "science": 92, "english": 85},
    {"name": "Dave",    "math": 55, "science": 61, "english": 59},
    {"name": "Eve",     "math": 98, "science": 95, "english": 91},
    {"name": "Frank",   "math": 63, "science": 70, "english": 77},
]

csv_path = Path("grades.csv")
with open(csv_path, "w", newline="") as f:
    fields = ["name", "math", "science", "english"]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(students)

print(f"Wrote {len(students)} student records to {csv_path}")


In [ ]:
# ── Step 2: Read back and compute stats ──────────────────────────────────────
def load_students(path):
    with open(path, "r") as f:
        reader = csv.DictReader(f)
        # Convert score columns to int
        records = []
        for row in reader:
            records.append({
                "name":    row["name"],
                "math":    int(row["math"]),
                "science": int(row["science"]),
                "english": int(row["english"]),
            })
    return records

def compute_stats(records):
    subjects = ["math", "science", "english"]
    stats = {}
    for subject in subjects:
        scores = [r[subject] for r in records]
        stats[subject] = {
            "average": round(sum(scores) / len(scores), 1),
            "highest": max(scores),
            "lowest":  min(scores),
        }

    # Add per-student average
    for r in records:
        r["average"] = round(sum(r[s] for s in subjects) / len(subjects), 1)

    return stats, records

records = load_students(csv_path)
stats, records = compute_stats(records)

print("Subject Statistics:")
for subject, s in stats.items():
    print(f"  {subject.capitalize():10s} avg={s['average']}, high={s['highest']}, low={s['lowest']}")

print("\nStudent Averages:")
for r in sorted(records, key=lambda x: x["average"], reverse=True):
    print(f"  {r['name']:<10s} {r['average']}")


In [ ]:
# ── Step 3: Save summary report to JSON ──────────────────────────────────────
report = {
    "total_students": len(records),
    "subject_stats":  stats,
    "top_student":    max(records, key=lambda r: r["average"])["name"],
    "class_average":  round(sum(r["average"] for r in records) / len(records), 1),
    "students":       records,
}

report_path = Path("grade_report.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Report saved to {report_path}")

# Verify by reading it back
with open(report_path) as f:
    loaded_report = json.load(f)

print(f"\nSummary from report file:")
print(f"  Total students: {loaded_report['total_students']}")
print(f"  Class average:  {loaded_report['class_average']}")
print(f"  Top student:    {loaded_report['top_student']}")
